In [28]:
import os
import pandas as pd
import emoji
from collections import Counter

# Leitura do dataset
df = pd.read_csv("../../../00-source/HateBR-main/dataset/HateBR.csv")
df.head()

,id,comentario,anotator1,anotator2,anotator3,label_final,links_post,account_post
0,1,Mais um lixo,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
1,2,Essa nao tem vergonha na cara!!,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
2,3,Essa mulher é doente.pilantra!,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
3,4,Comunista safada...,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
4,5,Vagabunda. Comunista. Mentirosa. O povo chilen...,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli


In [31]:
#Extração dos emojis dos textos
def extrair_emojis(texto):
    if pd.isna(texto):
        return []
    return [char for char in texto if char in emoji.EMOJI_DATA]

df["emojis"] = df["comentario"].apply(extrair_emojis_unicos)

# Identificação das mensagens de ódio
df_odio = df[df["label_final"] == 1] # 1 = discurso de ódio
df_nao_odio = df[df["label_final"] == 0] # 0 = não discurso de ódio

# Conta quantas mensagens de ódio possuem cada emoji
emojis_odio = Counter(
    e for conjunto in df_odio["emojis"] for e in conjunto
)

# Conta quantas mensagens de não ódio possuem cada emoji
emojis_nao_odio = Counter(
    e for conjunto in df_nao_odio["emojis"] for e in conjunto
)

# Junta todos os emojis encontrados nas duas categorias
todos = set(emojis_odio) | set(emojis_nao_odio)

# Montagem da tabela final
df_emojis = pd.DataFrame({
    "emoji": list(todos),
    "freq_odio": [emojis_odio.get(e, 0) for e in todos],
    "freq_nao_odio": [emojis_nao_odio.get(e, 0) for e in todos],
})

df_emojis["freq_total"] = (
    df_emojis["freq_odio"] + df_emojis["freq_nao_odio"]
)

# Normalização das frequências
total_odio = df_odio.shape[0]
total_nao_odio = df_nao_odio.shape[0]

df_emojis["freq_odio_norm"] = (
    df_emojis["freq_odio"] / total_odio
)

df_emojis["freq_nao_odio_norm"] = (
    df_emojis["freq_nao_odio"] / total_nao_odio
)

# Salvamento da tabela original
df_emojis.to_csv(
    "instagram_emojis_hatebr.csv",
    index=False,
    encoding="utf-8"
)

# Salvamento da tabela normalizada
df_emojis.to_csv(
    "instagram_emojis_normalizado.csv",
    index=False,
    encoding="utf-8"
)